# Customer Campaign Response Prediction
Binary classification on the Customer Personality Analysis dataset (target: `Response`).
Models: Logistic Regression, Decision Tree, kNN, Gaussian Naive Bayes, Random Forest.

**Before running:** `marketing_campaign.csv` must be in the same folder as this notebook.

## 1. Imports and setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix)

warnings.filterwarnings("ignore")
SEED = 42
os.makedirs("model", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 2. Load data and examine it
The file has an `.xls`/`.csv` name but is tab-separated text, so `sep='\t'` is used.

In [ ]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
print("Raw shape:", df.shape)
df.head()

In [ ]:
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

### Class distribution (imbalance check)

In [ ]:
print(df["Response"].value_counts())
print(df["Response"].value_counts(normalize=True).round(4))
sns.countplot(x="Response", data=df)
plt.title("Class distribution of target (Response)")
plt.savefig("results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Cleaning and feature engineering

In [ ]:
# constant columns carry no information; ID is only an identifier
df = df.drop(columns=["Z_CostContact", "Z_Revenue"])
dups = df.drop(columns=["ID"]).duplicated().sum()
print("Duplicate customers (ignoring ID):", dups)
df = df.drop(columns=["ID"]).drop_duplicates().reset_index(drop=True)

# engineered features (data was collected around 2014)
df["Age"] = 2014 - df["Year_Birth"]
df["Customer_Tenure_Days"] = (pd.Timestamp("2014-12-31") -
                              pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")).dt.days
df = df.drop(columns=["Year_Birth", "Dt_Customer"])

# remove impossible ages and tidy marital status labels
df = df[df["Age"] < 100]
df["Marital_Status"] = df["Marital_Status"].replace(
    {"Alone": "Single", "Absurd": "Other", "YOLO": "Other"})

X = df.drop(columns="Response")
y = df["Response"]
print("Shape after cleaning:", X.shape)

cat_cols = ["Education", "Marital_Status"]
num_cols = [c for c in X.select_dtypes(include="number").columns if c not in cat_cols]

## 4. Train-test split (80:20, stratified, fixed seed)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 4), "| Test:", round(y_test.mean(), 4))

# save the test data for the GitHub repo
pd.concat([X_test, y_test], axis=1).to_csv("test_data.csv", index=False)

## 5. Preprocessing (fitted on training data only, inside a Pipeline)

In [ ]:
def make_preprocessor(scale):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ])

# (needs_scaling, model) - scaling matters for LR / kNN / NB, not for trees
models = {
    "Logistic Regression": (True,  LogisticRegression(max_iter=1000, random_state=SEED)),
    "Decision Tree":       (False, DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=SEED)),
    "kNN":                 (True,  KNeighborsClassifier(n_neighbors=7)),
    "Naive Bayes":         (True,  GaussianNB()),
    "Random Forest (Ensemble)": (False, RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),
}

## 6. Train and evaluate all models

In [ ]:
rows, cms = [], {}
for name, (scale, clf) in models.items():
    pipe = Pipeline([("prep", make_preprocessor(scale)), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        "ML Model Name": name,
        "Accuracy":  accuracy_score(y_test, pred),
        "AUC":       roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall":    recall_score(y_test, pred),
        "F1":        f1_score(y_test, pred),
        "MCC":       matthews_corrcoef(y_test, pred),
    })
    cms[name] = confusion_matrix(y_test, pred)
    fname = name.split(" (")[0].lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipe, os.path.join("model", fname))

res = pd.DataFrame(rows).set_index("ML Model Name").round(4)
res.to_csv("results/metrics.csv")
res

## 7. Visualisations

In [ ]:
ax = res.plot(kind="bar", figsize=(12, 6), width=0.8)
ax.set_title("Model comparison across metrics"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
plt.xticks(rotation=15); plt.legend(loc="lower right", fontsize=8)
plt.savefig("results/metrics_by_model.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ax = res.T.plot(kind="bar", figsize=(12, 6), width=0.8)
ax.set_title("Evaluation metrics by model"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
plt.xticks(rotation=0); plt.legend(loc="lower right", fontsize=8)
plt.savefig("results/metrics_by_metric.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(res, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Metrics heatmap")
plt.savefig("results/metrics_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for a, (n, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a)
    a.set_title(n, fontsize=9); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.savefig("results/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()